# 12 — Ranking Top-K e métricas offline

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 12/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/12_metricas_ranking.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_12_metricas_ranking.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Calcular Precision@K, Recall@K e NDCG@K
- Definir conjunto de candidatos e relevância corretamente

## Fundamentação conceitual
Precision@K = acertos/K; Recall@K = acertos/número de itens relevantes; DCG@K pondera acertos pela posição $1/\log_2(j+1)$, e NDCG normaliza pelo melhor DCG possível. **São necessários rótulos de relevância por usuário e protocolo de candidatos.**

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Métricas em ranking controlado

In [2]:
def rank_metrics(recommendations,relevant,k=5):
    rec=list(recommendations)[:k]; rel=set(relevant)
    hits=[int(i in rel) for i in rec]
    precision=sum(hits)/k
    recall=sum(hits)/len(rel) if rel else 0.0
    dcg=sum(h/np.log2(j+2) for j,h in enumerate(hits))
    ideal=sum(1/np.log2(j+2) for j in range(min(len(rel),k)))
    return {'P@K':precision,'R@K':recall,'NDCG@K':dcg/ideal if ideal else 0.0}
print(rank_metrics(['A','B','C','D','E'],{'B','E','F'},5))

{'P@K': 0.4, 'R@K': 0.6666666666666666, 'NDCG@K': np.float64(0.4776237035032179)}


**Interpretação.** O exemplo é controlado: não presume que todas as ausências sejam negativas.

## Ranking de popularidade com verdade reservada

In [3]:
from sklearn.model_selection import train_test_split
tr,te=train_test_split(ratings,test_size=.2,random_state=42)
pop=tr.groupby('movieId').rating.agg(['mean','count']);pop['score']=(pop['mean']*pop['count']+20*tr.rating.mean())/(pop['count']+20)
user=te.loc[te.rating.ge(4),'userId'].value_counts().index[0]
seen=set(tr.loc[tr.userId.eq(user),'movieId']); relevant=set(te.loc[(te.userId.eq(user)) & (te.rating.ge(4)),'movieId'])
recs=[i for i in pop.sort_values('score',ascending=False).index if i not in seen][:10]
print('Usuário:',user,'relevantes observados:',len(relevant),rank_metrics(recs,relevant,k=10))

Usuário: 414 relevantes observados: 259 {'P@K': 0.9, 'R@K': 0.03474903474903475, 'NDCG@K': np.float64(0.9363792118010483)}


**Interpretação.** A medida é **parcial**: o teste observa apenas alguns itens avaliados, não a relevância de todo o catálogo.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** ranking e negativos. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **ranking e negativos** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Precision@K e Recall@K com casos extremos

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [4]:
def metricas_topk(recomendados,relevantes,k):
 topo=recomendados[:k];inter=len(set(topo)&set(relevantes))
 return inter/k,inter/len(set(relevantes)) if relevantes else 0.
print('Usuário com relevantes:',metricas_topk([3,4,5],[3,5],2))
print('Usuário sem relevantes:',metricas_topk([3,4,5],[],2))

Usuário com relevantes: (0.5, 0.5)
Usuário sem relevantes: (0.0, 0.0)


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **ranking e negativos** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Interpretação das métricas de Top-K

Com relevância binária, `DCG@K = Σ_{j=1..K} rel_j / log₂(j+1)` e `NDCG@K = DCG@K/IDCG@K`. Precision@K usa denominador `K` na convenção aplicada. O resultado depende do **conjunto de candidatos**, do critério de relevância e da exposição observável. Itens não avaliados não podem ser assumidos negativos verdadeiros.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Verificar limites das métricas e posição dos acertos
ex = rank_metrics(['A','B','C','D','E'], {'B','E','F'}, k=5)
print('Exemplo controlado:', ex)
assert all(0.0 <= valor <= 1.0 for valor in ex.values())
assert rank_metrics(['B','E','A','C','D'], {'B','E','F'}, 5)['NDCG@K'] > ex['NDCG@K']

Exemplo controlado: {'P@K': 0.4, 'R@K': 0.6666666666666666, 'NDCG@K': np.float64(0.4776237035032179)}


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

<!-- EP_FASE2_09_LOTE1 -->
### Protocolo complementar — usuários definidos pelo treino e agregação

O exemplo anterior escolhe um usuário **pelas notas positivas no teste**: é uma ilustração, não avaliação agregada. Abaixo, selecionamos usuários por um critério estabelecido **no treino** (pelo menos cinco avaliações no histórico), separamos os últimos 20% **por usuário** e ajustamos a popularidade suavizada apenas com os dados do treino. O catálogo candidato é o conjunto de itens observados no treino, excluindo os itens já avaliados pelo próprio usuário.

**Interpretação:** como MovieLens não informa exposição nem relevância dos itens não avaliados, o Precision@K calculado em relação aos positivos observados é **um proxy incompleto**, não a precisão real de uma plataforma. Recall e NDCG não são definidos para usuários sem positivos observados no holdout; reportamos a quantidade de usuários excluídos e a taxa de cobertura. O corte é cronológico dentro de cada usuário, mas **não garante um instante global único de treino**, e não há seleção de hiperparâmetros nem estimativa causal.


In [6]:
# Separação cronológica POR USUÁRIO, reproduzível; nenhuma nota do holdout entra no ranking.
ep_ordenado = ratings.sort_values(['userId', 'timestamp', 'movieId']).copy()
ep_ordem = ep_ordenado.groupby('userId').cumcount().to_numpy()
ep_total = ep_ordenado.groupby('userId').userId.transform('size').to_numpy()
ep_n_holdout = np.maximum(1, np.floor(ep_total * 0.20).astype(int))
ep_e_teste = ep_ordem >= ep_total - ep_n_holdout
ep_tr = ep_ordenado.loc[~ep_e_teste]
ep_te = ep_ordenado.loc[ep_e_teste]
ep_pop = ep_tr.groupby('movieId').rating.agg(['mean', 'count'])
ep_media_global = float(ep_tr.rating.mean())
ep_pop['score'] = (ep_pop['mean'] * ep_pop['count'] + 20 * ep_media_global) / (ep_pop['count'] + 20)
ep_ranking_global = ep_pop.sort_values('score', ascending=False).index.tolist()
ep_treino_usuario = ep_tr.groupby('userId').movieId.agg(set)
ep_testes_positivos = ep_te.loc[ep_te.rating.ge(4)].groupby('userId').movieId.agg(set)
ep_usuarios_elegiveis = sorted(
    uid for uid, itens in ep_treino_usuario.items() if len(itens) >= 5
)
ep_resultados = []
ep_sem_positivos = 0
for ep_uid in ep_usuarios_elegiveis:
    ep_vistos = ep_treino_usuario[ep_uid]
    ep_relevantes = ep_testes_positivos.get(ep_uid, set()) & set(ep_ranking_global)
    if not ep_relevantes:
        ep_sem_positivos += 1
        continue
    ep_top10 = [i for i in ep_ranking_global if i not in ep_vistos][:10]
    ep_metricas = rank_metrics(ep_top10, ep_relevantes, k=10)
    ep_resultados.append({'userId': ep_uid, 'positivos_teste_observados': len(ep_relevantes),
                          **ep_metricas})
ep_tabela = pd.DataFrame(ep_resultados)
print('Usuários elegíveis (definidos apenas no treino):', len(ep_usuarios_elegiveis))
print('Sem positivos observados no teste (fora da média):', ep_sem_positivos)
print('Usuários com Recall/NDCG observáveis:', len(ep_tabela))
if not ep_tabela.empty:
    print('Médias condicionais em usuários com positivos observados — PROXIES:')
    print(ep_tabela[['P@K', 'R@K', 'NDCG@K']].mean().round(4).to_string())
    print('Cobertura de avaliação:', f'{len(ep_tabela)/len(ep_usuarios_elegiveis):.1%}')
assert len(ep_tabela) + ep_sem_positivos == len(ep_usuarios_elegiveis)


Usuários elegíveis (definidos apenas no treino): 610
Sem positivos observados no teste (fora da média): 20
Usuários com Recall/NDCG observáveis: 590
Médias condicionais em usuários com positivos observados — PROXIES:
P@K       0.0495
R@K       0.0406
NDCG@K    0.0641
Cobertura de avaliação: 96.7%


## Limitações e boas práticas
Não compare resultados de estudos com diferentes conjuntos de candidatos, cortes temporais ou critérios de relevância.

## Exercícios
Explique por que a avaliação com negativos não observados pode superestimar desempenho.

## Soluções comentadas
Sem exposição conhecida, itens não avaliados não são negativos verdadeiros; é preciso explicitar a estratégia de amostragem.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [7]:
# Exportação de resultados do capítulo
# Execute todas as células em sequência antes de exportar.
# Cada arquivo documenta um experimento ou protocolo distinto.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"userId": user, "k": 10, "n_relevantes_observados": len(relevant),
               **rank_metrics(recs, relevant, k=10)}]).to_csv(
    out / "12_metricas_topk_exemplo.csv", index=False)

# Resultados das análises metodológicas complementares
_ep_destino = out / '12_metricas_topk_por_usuario_temporal.csv'
_ep_df = ep_tabela.copy()
if _ep_df.empty:
    _ep_df = pd.DataFrame(columns=['userId', 'positivos_teste_observados', 'P@K', 'R@K', 'NDCG@K'])
_ep_df.assign(k=10, tipo_resultado='proxy_offline_condicional').to_csv(_ep_destino, index=False)
_ep_avaliados = len(ep_tabela)
_ep_medias = ep_tabela[['P@K', 'R@K', 'NDCG@K']].mean() if _ep_avaliados else pd.Series(dtype=float)
_ep_destino2 = out / '12_resumo_topk_temporal.csv'
pd.DataFrame([{
    'k': 10, 'usuarios_elegiveis_por_treino': int(len(ep_usuarios_elegiveis)),
    'usuarios_sem_positivos_observados': int(ep_sem_positivos),
    'usuarios_avaliados': int(_ep_avaliados),
    'cobertura_da_avaliacao': (_ep_avaliados / len(ep_usuarios_elegiveis)) if ep_usuarios_elegiveis else None,
    'precision_media_condicional': float(_ep_medias['P@K']) if _ep_avaliados else None,
    'recall_media_condicional': float(_ep_medias['R@K']) if _ep_avaliados else None,
    'ndcg_media_condicional': float(_ep_medias['NDCG@K']) if _ep_avaliados else None,
    'protocolo': 'holdout_cronologico_por_usuario;positivos_observados;sem_logs_exposicao'
}]).to_csv(_ep_destino2, index=False)
print('Arquivos exportados:', _ep_destino, _ep_destino2)


Arquivos exportados: outputs/12_metricas_topk_por_usuario_temporal.csv outputs/12_resumo_topk_temporal.csv
